# Notebook 02: Data Download

**Purpose:** Download CORU dataset from Hugging Face and prepare for splitting.

This notebook will:
1. Load CORU dataset from Hugging Face
2. Subset to 400-500 images
3. Extract images and annotations
4. Validate data
5. Generate dataset summary

In [ ]:
# Step 1: Load CORU dataset
from datasets import load_dataset
import json
from pathlib import Path
from PIL import Image
import io

print("Loading CORU dataset from Hugging Face...")
dataset = load_dataset("abdoelsayed/CORU")

print(f"Dataset loaded. Available splits: {list(dataset.keys())}")
for split in dataset:
    print(f"  {split}: {len(dataset[split])} samples")

print("\nFirst 3 examples:")
for i, example in enumerate(dataset['train'].take(3)):
    print(f"  Example {i}: keys = {list(example.keys())}")

In [ ]:
# Step 2: Subset to 400-500 images
import random

random.seed(42)
target_size = 450
dataset_size = len(dataset['train'])

if dataset_size > target_size:
    # Randomly select target_size samples
    indices = random.sample(range(dataset_size), target_size)
    subset = dataset['train'].select(indices)
    print(f"Selected {len(subset)} images from {dataset_size} total")
else:
    subset = dataset['train']
    print(f"Using all {len(subset)} images (less than target {target_size})")

print(f"✓ Subset created: {len(subset)} images")

In [ ]:
# Step 3: Extract images and annotations
import os

output_dir = Path("/content/docuspend/data/raw/all") if os.path.exists("/content") else Path("./docuspend/data/raw/all")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving images to: {output_dir}")

coru_annotations = {}
saved_count = 0

for idx, example in enumerate(subset):
    filename = f"receipt_{idx:06d}.jpg"
    
    # Save image
    if 'image' in example:
        image = example['image']
        if isinstance(image, dict):  # Potential dict format
            img_data = image.get('bytes')
            if img_data:
                with open(output_dir / filename, 'wb') as f:
                    f.write(img_data)
        else:
            image.save(output_dir / filename)
        saved_count += 1
    
    # Store annotation
    annotation = {k: v for k, v in example.items() if k != 'image'}
    coru_annotations[filename] = annotation
    
    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx + 1}/{len(subset)}")

print(f"\n✓ Saved {saved_count} images")

In [ ]:
# Save annotations
annotations_file = output_dir / "coru_annotations_raw.json"
with open(annotations_file, 'w') as f:
    json.dump(coru_annotations, f, indent=2)

print(f"✓ Saved {len(coru_annotations)} annotations to {annotations_file}")

In [ ]:
# Step 4: Data validation
from PIL import Image

print("Validating downloaded data...")

valid_count = 0
corrupted = []
format_counts = {}
sizes = []

for img_file in sorted(output_dir.glob("receipt_*.jpg")):
    try:
        img = Image.open(img_file)
        img.verify()
        valid_count += 1
        
        img = Image.open(img_file)  # Reopen after verify
        fmt = img.format
        format_counts[fmt] = format_counts.get(fmt, 0) + 1
        sizes.append(img.size)
        
    except Exception as e:
        corrupted.append((img_file.name, str(e)))

print(f"✓ Valid images: {valid_count}")
print(f"✗ Corrupted: {len(corrupted)}")
print(f"\nFormats: {format_counts}")
print(f"Image dimensions: {min(sizes)} to {max(sizes)}")

In [ ]:
# Step 5: Generate dataset summary
import os

total_size_mb = sum(os.path.getsize(f) for f in output_dir.glob("*.jpg")) / (1024 * 1024)
avg_size_mb = total_size_mb / valid_count if valid_count > 0 else 0

summary = {
    "source": "CORU (Hugging Face)",
    "total_images": valid_count,
    "total_size_mb": round(total_size_mb, 2),
    "avg_image_size_mb": round(avg_size_mb, 2),
    "image_formats": format_counts,
    "date_downloaded": str(pd.Timestamp.now().date()),
    "license": "CC-BY-4.0",
    "status": "ready_for_preprocessing"
}

summary_file = Path("/content/docuspend/data/dataset_info.json") if os.path.exists("/content") else Path("./docuspend/data/dataset_info.json")
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print("Dataset Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")
print(f"\n✓ Summary saved to {summary_file}")

In [ ]:
# Step 6: Output status report
print("="*60)
print("DATA DOWNLOAD COMPLETE")
print("="*60)
print(f"✅ Downloaded {valid_count} images from CORU")
print(f"✅ Total dataset size: {total_size_mb:.1f} MB")
print(f"✅ Images saved to: {output_dir}")
print(f"✅ Annotations saved")
print(f"✅ Ready for train/val/test split")
print(f"\nNext: Run 03_data_preparation.ipynb")